# NYC Bike Data: Dataset Description & Exploratory Analysis

This notebook documents and explores every dataset used in the project and underpins the
**technical report** in `docs/technical-report/`. It covers, in order of importance:

1. **Citi Bike dataset**: the project's main data source with bike trip records (one monthly file, downloaded manually).
2. **Station metadata**: current Citi Bike stations from the Lyft GBFS feed (fetched live).
3. **Weather**: hourly NYC observations from the Open-Meteo archive (fetched live).
4. **Bike lanes**: NYC bike-route network from NYC OpenData (fetched live).

For each dataset we describe its **structure**, run **exploratory data analysis (EDA)**, and
assess **data quality**. See `src/notebooks/README.md` for how to obtain the ride data and run
this notebook.

## 0. Setup

In [ ]:
import io
import json
import math
import re
from datetime import date, timedelta
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import requests
import folium
from folium.plugins import HeatMap
from matplotlib.collections import LineCollection
from matplotlib.lines import Line2D
from PIL import Image

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)

# Where to find the manually-downloaded Citi Bike monthly CSV(s) (relative to this notebook).
DATA_DIR = Path("data")

# Data sources (mirrors src/ingestion/config.yaml and src/backend/config.yaml).
WEATHER_API_URL = "https://archive-api.open-meteo.com/v1/archive"
BIKE_ROUTES_URL = "https://data.cityofnewyork.us/api/views/mzxg-pwib/rows.csv?accessType=DOWNLOAD"
GBFS_INFO_URL = "https://gbfs.lyft.com/gbfs/2.3/bkn/en/station_information.json"
GBFS_STATUS_URL = "https://gbfs.lyft.com/gbfs/2.3/bkn/en/station_status.json"
NYC_LAT, NYC_LON = 40.7823234, -73.9654161
NYC_TZ = "America/New_York"

# Distance-feature constants (see src/ingestion/sources/distances.py).
EARTH_RADIUS_KM = 6371
STREET_CIRCUITY_FACTOR = 1.3

Here we define an utility function for computing the Haversine distance calculated as:

$$d = 2 r \arcsin \sqrt{\sin^2\left(\frac{\Delta \phi}{2}\right) + \cos(\phi_1) \cos(\phi_2) \sin^2\left(\frac{\Delta \lambda}{2}\right)}$$

where $\phi$ is latitude, $\lambda$ is longitude, and $r$ is the Earth's radius (6371 km).

In [ ]:
# Vectorised great-circle distance (km) scaled by the street-circuity factor,
# matching the production formula in src/ingestion/sources/distances.py.
def haversine_km(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat, dlon = lat2 - lat1, lon2 - lon1
    a = np.sin(dlat / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    c = 2 * np.arcsin(np.sqrt(a))
    return EARTH_RADIUS_KM * c * STREET_CIRCUITY_FACTOR

## 1. Citi Bike Trip Data

The core dataset Citi Bike publishes one CSV per month, each row a single trip, available from
the [system data page](https://www.citibikenyc.com/system-data). Download one month and place the
extracted CSV in the `data/` folder next to this notebook (see `README.md`). The analysis shown here uses the **February 2025** file (`202502-citibike-tripdata`).

Newer files (2020+) and the legacy (pre-2020) schema use different column names. This map normalises the legacy columns to the modern schema so the rest of the notebook works regardless of which month was downloaded.

In [ ]:
LEGACY_RENAME = {
    "starttime": "started_at",
    "stoptime": "ended_at",
    "start station name": "start_station_name",
    "start station id": "start_station_id",
    "end station name": "end_station_name",
    "end station id": "end_station_id",
    "start station latitude": "start_lat",
    "start station longitude": "start_lng",
    "end station latitude": "end_lat",
    "end station longitude": "end_lng",
}

def load_rides(data_dir: Path):
    csv_files = sorted(data_dir.glob("*.csv"))
    if not csv_files:
        raise FileNotFoundError(
            f"No CSV files found in {data_dir.resolve()}. Download a Citi Bike monthly "
            "trip file, extract it, and place the CSV here. See src/notebooks/README.md."
        )
    df = pd.concat([pd.read_csv(f, low_memory=False) for f in csv_files], ignore_index=True)

    # Columns unique to the legacy schema — kept track of for the data-quality section.
    legacy_cols = [c for c in ["gender", "birth year", "bikeid", "tripduration", "usertype"]
                   if c in df.columns]
    is_legacy = "rideable_type" not in df.columns
    if is_legacy:
        df = df.rename(columns=LEGACY_RENAME)
        if "usertype" in df.columns:
            df["member_casual"] = df["usertype"].map(
                {"Subscriber": "member", "Customer": "casual"}
            ).fillna(df["usertype"])
        df["rideable_type"] = pd.NA          # absent before the e-bike rollout
        if "ride_id" not in df.columns:
            df["ride_id"] = df.index.astype(str)

    # Recent months carry fractional seconds meanwhile older months do not. Use a mixed format to handle both.
    for col in ["started_at", "ended_at"]:
        try:
            df[col] = pd.to_datetime(df[col], format="ISO8601")
        except (ValueError, TypeError):
            df[col] = pd.to_datetime(df[col], format="mixed")
    return df, {"files": [f.name for f in csv_files], "is_legacy": is_legacy,
                "legacy_cols": legacy_cols}

df, meta = load_rides(DATA_DIR)
print("Loaded files :", meta["files"])
print("Schema       :", "legacy (pre-2020)" if meta["is_legacy"] else "modern (2020+)")
print("Shape        :", df.shape)
df.head()

### 1.1 Structure

We provide a brief overview of the dataset's structure, including column names, data types, and sample records, through the `pandas` library. 

In [ ]:
df.info()

print("\nUnique rideable_type :", df["rideable_type"].dropna().unique().tolist())
print("Unique member_casual :", df["member_casual"].dropna().unique().tolist())

unique_stations = pd.concat([df["start_station_name"], df["end_station_name"]]).nunique()
print(f"Unique stations      : {unique_stations}")
print(f"Date range           : {df['started_at'].min()} -> {df['started_at'].max()}")

### 1.2 Trip Duration

Given the raw trip data, we compute a set of derived features, including trip duration and distance. This is done in order to enhance the analysis of user behaviour and trip patterns. 

In [ ]:
df["ride_length_s"] = (df["ended_at"] - df["started_at"]).dt.total_seconds()
df["ride_length_min"] = df["ride_length_s"] / 60

print("Ride length (minutes) summary:")
print((df["ride_length_s"] / 60).agg(["mean", "median", "min", "max", "std"]).round(2))

negative = (df["ride_length_s"] < 0).sum()
long_rides = (df["ride_length_s"] > 4 * 3600).sum()
print(f"\nRides with negative duration : {negative}")
print(f"Rides longer than 4 hours    : {long_rides}")

# Every trip is kept
plt.figure(figsize=(8, 5))
sns.histplot(df["ride_length_min"].clip(lower=0, upper=60), bins=60)
plt.axvline(df["ride_length_min"].median(), color="red", ls="--",
            label=f"median {df['ride_length_min'].median():.1f} min")
plt.title("Ride length (0\u201360 min)")
plt.xlabel("Ride length (min)")
plt.legend()
plt.tight_layout()
plt.show()

Ride length is strongly **right-skewed**: most trips last only a few to ~20 minutes (the
histogram is zoomed to the first hour), with a thin tail stretching to much longer rides. No
trips are excluded here. The few **negative** durations (clock/data glitches where a trip ends
before it starts) and rides over **4 hours** (more likely a bike that was never docked correctly
than a genuine 4-hour trip) are kept and folded into the edge bins, and are quantified above. The
distribution starts at 1 minute because Citi Bike removes trips shorter than 60 seconds at the
source (likely false starts or re-docking), so there are no zero-length trips.

### 1.3 Trip Distance

In [ ]:
# Straight-line distance between start and end stations, scaled by the circuity factor
df["trip_distance_km"] = haversine_km(
    df["start_lat"], df["start_lng"], df["end_lat"], df["end_lng"]
)

print("Trip distance (km) summary:")
print(df["trip_distance_km"].agg(["mean", "median", "min", "max", "std"]))

# Nothing is excluded
cap = 10  # km; distances above this collapse into the last bin
plt.figure(figsize=(8, 5))
sns.histplot(df["trip_distance_km"].clip(upper=cap), bins=60)
plt.xlabel("Trip distance (km)")
plt.title("Trip distance (all trips; tail >= 10 km folded)")
plt.tight_layout()
plt.show()

This is the **straight-line** distance between the start and end stations, scaled by a 1.3
circuity factor to approximate the real on-street path. No trips are excluded: **round trips**
that begin and end at the same station appear as the **0 km** bar (the straight-line metric
cannot capture their real path), and the long-distance tail is folded into the final bin
so these anomalies stay visible. Both must be considered when interpreting the distribution or
computing distance-based statistics (e.g., average distance per trip, average speed).

### 1.4 Bike Type and User Type

The objective of this analysis is to understand the relationship between bike type and user type. The first panel shows the **bike-type mix within each user type**. Meanwhile the second panel displays the **median ride length by user and bike type**.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Bike-type mix *within* each user type (share, not raw volume)
counts = df.groupby(["member_casual", "rideable_type"]).size().reset_index(name="rides")
counts["share"] = 100 * counts["rides"] / counts.groupby("member_casual")["rides"].transform("sum")
sns.barplot(data=counts, x="member_casual", y="share", hue="rideable_type", ax=axes[0])
axes[0].set(title="Bike-type mix by user type", xlabel="User type",
            ylabel="Share of the user's rides (%)")

# Median ride length by user and bike type
med = (df.groupby(["member_casual", "rideable_type"])["ride_length_min"]
         .median().reset_index())
sns.barplot(data=med, x="member_casual", y="ride_length_min", hue="rideable_type", ax=axes[1])
axes[1].set(title="Median ride length by user & bike type", xlabel="User type",
            ylabel="Median ride length (min)")

# Add headroom and pin each legend to the top-right so it never overlaps a bar.
for ax in axes:
    ax.set_ylim(0, ax.get_ylim()[1] * 1.3)
    ax.legend(title="rideable_type", loc="upper right", framealpha=0.9)

plt.tight_layout()
plt.show()

# Overall composition, for context.
print("Share of rides by user type (%):")
print((df["member_casual"].value_counts(normalize=True) * 100).round(1))
print("\nShare of rides by bike type (%):")
print((df["rideable_type"].value_counts(normalize=True) * 100).round(1))

Raw volume is dominated by **members** (commuter use), but that count alone is not very
informative. Splitting each dimension by the other is more revealing. **Bike-type mix** (left):
**casual** riders lean more heavily on **electric** bikes than members do. **Ride length**
(right): casual riders take **longer** trips than members on both bike types, while members ride a
steady, short duration regardless of bike type, consistent with frequent short-hop commuting versus
occasional, mostly-electric leisure trips.

### 1.5 Temporal Patterns

Our purpose here is to explore the temporal patterns of bike usage, including daily and weekly trends, through the ride counts. 

In [ ]:
# Hour and weekday are taken from the END timestamp, matching the ingestion pipeline
df["hour"] = df["ended_at"].dt.hour
df["day_of_week"] = df["ended_at"].dt.day_name()
df["date"] = df["ended_at"].dt.date
df["day_type"] = np.where(
    df["ended_at"].dt.weekday < 5, "Weekday", "Weekend"
)

# Mean rides per hour, separating weekdays from weekends.
hourly = (
    df.groupby(["date", "hour", "day_type"]).size().reset_index(name="rides")
)
mean_rides = hourly.groupby(["hour", "day_type"])["rides"].mean().reset_index()

plt.figure(figsize=(12, 5))
sns.barplot(x="hour", y="rides", hue="day_type", data=mean_rides)
plt.xlabel("Hour of day")
plt.ylabel("Mean rides per day")
plt.title("Mean rides by hour: weekday vs weekend")
plt.tight_layout()
plt.show()

Weekdays show the classic bimodal commuting pattern (peaks around 8 AM and 5-6 PM), while
weekends follow a single midday hump consistent with leisure use.

### 1.6 Spatial Patterns

We explore the spatial patterns of bike usage, using a spatial heatmap to visualize the distribution of bike trips across the city. This analysis helps identify areas with high demand for bike-sharing services.

In [ ]:
# Heatmap of trip origins, weighted by how many trips start at each station.
origins = (
    df.dropna(subset=["start_lat", "start_lng", "start_station_name"])
    .groupby("start_station_name")
    .agg(lat=("start_lat", "first"), lon=("start_lng", "first"), rides=("ride_id", "count"))
    .reset_index()
)
bike_map = folium.Map(location=[origins["lat"].mean(), origins["lon"].mean()], zoom_start=12)
HeatMap(origins[["lat", "lon", "rides"]].values.tolist()).add_to(bike_map)
# recreate the map and add a tighter heatmap to reduce the glow
bike_map = folium.Map(location=[origins["lat"].mean(), origins["lon"].mean()], zoom_start=12)
HeatMap(
    origins[["lat", "lon", "rides"]].values.tolist(),
    radius=8,
    blur=6,
    min_opacity=0.3,
    max_zoom=18
).add_to(bike_map)
bike_map

In [ ]:
# Station-to-station flow: how many trips run between each ordered station pair.
flow = (
    df.dropna(subset=["start_station_name", "end_station_name"])
    .groupby(["start_station_name", "end_station_name"]).size()
    .reset_index(name="trips")
)
print(f"Distinct station pairs with at least one trip: {len(flow):,}")
print("\nTop 10 station-to-station flows:")
print(flow.sort_values("trips", ascending=False).head(10).to_string(index=False))

Demand is highly concentrated: a small set of stations near transit hubs and the business
districts dominate both departures and arrivals, so the heatmap clusters around Midtown and Lower
Manhattan, and the busiest station-to-station flows tend to link these same hubs.

### 1.7 Data Quality

#### Missing Values

A small share of trips have missing values, and they are confined to the **station** columns.
The **end-of-trip** fields (`end_station_name`, `end_station_id`, `end_lat`, `end_lng`) are the most
affected probably due to trips ending away from a station. The non-station columns (`ride_id`, `rideable_type`, `started_at`,
`ended_at`, `member_casual`) have no missing values. The table below reports the missing count and
percentage for every raw column.

In [ ]:
raw_cols = ["ride_id", "rideable_type", "started_at", "ended_at",
            "start_station_name", "start_station_id", "end_station_name", "end_station_id",
            "start_lat", "start_lng", "end_lat", "end_lng", "member_casual"]
missing = df[raw_cols].isna().sum()
missing = (pd.DataFrame({"missing": missing, "pct (%)": (missing / len(df) * 100).round(2)})
           .sort_values("missing", ascending=False))
print(f"{len(df):,} trips loaded; missing values per column:")
missing

#### Trip Outliers

A handful of trips are clearly not ordinary rides. Some are **too long**
which points to a bike that was never returned rather than a continuous ride. Others cover **zero
straight-line distance** because they start and end at the same station (round trips the
point-to-point distance cannot capture). A few are **impossibly far** caused by
broken station coordinates (an endpoint at latitude/longitude 0, 0). The table below shows
examples of each.

In [ ]:
cols = ["started_at", "ride_length_min", "trip_distance_km",
        "start_station_name", "end_station_name", "member_casual"]
too_long = (df[(df["trip_distance_km"] > 0) & df["end_station_name"].notna()]
            .nlargest(3, "ride_length_min")[cols].assign(issue="too long"))
zero_dist = (df[df["trip_distance_km"] == 0]
             .nlargest(3, "ride_length_min")[cols].assign(issue="zero distance (round trip)"))
too_far = (df[df["trip_distance_km"] > 4000]
           .nlargest(2, "trip_distance_km")[cols].assign(issue="too far (bad coordinates)"))
print(f"trips over 24 h           : {int((df['ride_length_min'] > 24 * 60).sum()):,}")
print(f"zero-distance round trips : {int((df['trip_distance_km'] == 0).sum()):,}")
print(f"trips over 4000 km        : {int((df['trip_distance_km'] > 4000).sum()):,}")
pd.concat([too_long, zero_dist, too_far]).round({"ride_length_min": 0, "trip_distance_km": 2})

#### Different Structure

Citi Bike's trip-file schema changed around **2020**. Files from **2013-2019** (and the Jersey
City `JC-*` files) use a **legacy** layout: space-separated column names, a `bikeid`, a `usertype`
of `Subscriber`/`Customer`, and rider demographics (`gender`, `birth year`); they carry no
`ride_id` and no `rideable_type` (electric bikes did not exist yet). From **2020** onward the
files use the **modern** schema analysed above.


In [ ]:
# Compare the two actual schemas: the modern layout (2020+, loaded above) and the legacy
# pre-2020 layout that `load_rides` maps onto it via LEGACY_RENAME.
MODERN = ["ride_id", "rideable_type", "started_at", "ended_at",
          "start_station_name", "start_station_id", "end_station_name", "end_station_id",
          "start_lat", "start_lng", "end_lat", "end_lng", "member_casual"]
assert all(c in df.columns for c in MODERN)  # these are the loaded modern file's raw columns

modern_to_legacy = {v: k for k, v in LEGACY_RENAME.items()}
modern_to_legacy["member_casual"] = "usertype"
notes = {
    "ride_id": "added - unique trip id",
    "rideable_type": "added - classic vs electric (e-bikes)",
    "member_casual": "renamed + recoded (Subscriber->member, Customer->casual)",
}
rows = [{"Modern (2020+)": c,
         "Legacy (2013-2019)": modern_to_legacy.get(c, "-"),
         "Change": notes.get(c, "renamed")}
        for c in MODERN]
for leg, note in [("tripduration", "dropped - derivable from timestamps"),
                  ("bikeid", "dropped - physical bike id"),
                  ("birth year", "dropped - rider demographic"),
                  ("gender", "dropped - rider demographic")]:
    rows.append({"Modern (2020+)": "-", "Legacy (2013-2019)": leg, "Change": note})

print(f"Loaded month uses the {'LEGACY' if meta['is_legacy'] else 'MODERN'} schema.")
pd.DataFrame(rows)

## 2. Station Metadata (GBFS)

Current station information and live availability from the Lyft GBFS feed (the same feed the
backend uses). Two endpoints are merged based on `station_id`: `station_information` (static: name, location, capacity)
and `station_status` (live: bikes/docks available, operational flags).

In [ ]:
info = requests.get(GBFS_INFO_URL, timeout=(5, 30)).json()["data"]["stations"]
status = requests.get(GBFS_STATUS_URL, timeout=(5, 30)).json()["data"]["stations"]
status_map = {s["station_id"]: s for s in status}

rows = []
for s in info:
    st = status_map.get(s["station_id"], {})
    counts = {v.get("vehicle_type_id"): v.get("count", 0)
              for v in st.get("vehicle_types_available", [])}
    rows.append({
        "station_id": s["station_id"],
        "short_name": s.get("short_name"),
        "name": s.get("name"),
        "lat": s.get("lat"),
        "lon": s.get("lon"),
        "capacity": s.get("capacity"),
        "num_bikes_available": st.get("num_bikes_available"),
        "num_classic": counts.get("1"),
        "num_ebikes": counts.get("2"),
        "num_ebikes_reported": st.get("num_ebikes_available"),
        "num_docks_available": st.get("num_docks_available"),
        "num_bikes_disabled": st.get("num_bikes_disabled"),
        "num_docks_disabled": st.get("num_docks_disabled"),
        "is_installed": st.get("is_installed"),
        "is_renting": st.get("is_renting"),
        "is_returning": st.get("is_returning"),
    })
stations = pd.DataFrame(rows)
stations["active"] = (
    (stations["is_installed"] == 1) & (stations["is_renting"] == 1) & (stations["is_returning"] == 1)
)
print(f"Stations in feed: {len(stations)}  (active: {stations['active'].sum()})")

# The feed lists not-installed stations first, so a plain head() would show only those.
preview = pd.concat([stations[stations["active"]].head(3),
                     stations[~stations["active"]].head(2)])
preview

### 2.1 Structure

We provide a brief overview of the merged table's structure, including column names, data types, and sample records, through the `pandas` library.

Two **extracted features** are derived on top of the raw feed fields. `active` combines the three operational flags (`is_installed`, `is_renting`, `is_returning`) into a single boolean, used to filter unavailable stations out of live views. `num_classic` and `num_ebikes` split the bikes at a station by vehicle type: the feed reports availability as a list of `(vehicle_type_id, count)` pairs, so it is indexed by type id (`"1"` classic, `"2"` electric), and a type absent from that list leaves no count.

In [ ]:
stations.info()

print("\nExtracted features (active and inactive stations):")
print(preview[["short_name", "num_bikes_available", "num_classic", "num_ebikes",
               "is_installed", "is_renting", "is_returning", "active"]])

### 2.2 Station Capacity

We look at how the network is provisioned, through the **distribution of station capacity**: how many docks each station has, and how many stations share a given size.

In [ ]:
plt.figure(figsize=(8, 4))
sns.histplot(stations["capacity"].dropna(), bins=30)
plt.title("Station capacity")
plt.xlabel("Docks per station")
plt.ylabel("Number of stations")
plt.tight_layout()
plt.show()

print(f"Stations                  : {len(stations)}")
print(f"Median docks per station  : {stations['capacity'].median():.0f}")
print(f"Stations with capacity 0  : {int((stations['capacity'] == 0).sum())}")

Capacity distinguishes **small stations** from the **larger hubs** that anchor the busiest areas, which is what the dashboard encodes when sizing station markers. The bar at **0 docks** is not corrupt data: those are stations that appear in the feed but are not physically installed.

### 2.3 Live Availability

We now turn to the current state of the network. Rather than a few headline totals, the chart decomposes the network's **entire capacity**, so every dock the feed knows about is counted exactly once, in one of six states: holding a rentable **classic bike** or **e-bike**, **free** to receive one, holding a bike that is **out of service**, being a **dock** that is itself out of service, or **unreported**, meaning capacity that no counter accounts for. The same breakdown is repeated for active and inactive stations, whose composition turns out to be entirely different.

In [ ]:
# Live snapshot: these totals change on every run.
def dock_breakdown(d):
    b = pd.Series({
        "Classic bikes": d["num_classic"].sum(),
        "E-bikes": d["num_ebikes"].sum(),
        "Free docks": d["num_docks_available"].sum(),
        "Bikes out of service": d["num_bikes_disabled"].sum(),
        "Docks out of service": d["num_docks_disabled"].sum(),
    })
    b["Unreported"] = d["capacity"].sum() - b.sum()  # capacity no counter accounts for
    return b

groups = {
    "Whole network": stations,
    "Active stations": stations[stations["active"]],
    "Inactive stations": stations[~stations["active"]],
}
parts = pd.DataFrame({k: dock_breakdown(d) for k, d in groups.items()}).T
share = 100 * parts.div(parts.sum(axis=1), axis=0)

colors = dict(zip(share.columns,
                  ["#2a7f62", "#7fc4a8", "#4c78a8", "#e07b39", "#b0561a", "#bdbdbd"]))
STEP = 1.7  # vertical spacing between bars, leaving room for the call-outs around each
fig, ax = plt.subplots(figsize=(10, 5))
for i, g in [(n * STEP, g) for n, g in enumerate(share.index)]:
    left, thin = 0.0, 0
    for col in share.columns:
        s, n = share.loc[g, col], parts.loc[g, col]
        ax.barh(i, s, left=left, color=colors[col], height=0.6,
                label=col if i == 0 else None)
        if s >= 4:   # wide enough to hold the value inside the segment
            ax.text(left + s / 2, i, f"{s:.0f}%\n{int(n):,}", ha="center", va="center",
                    fontsize=9, color="white")
        elif n > 0:  # too thin to write in: call it out beside the bar
            x = left + s / 2
            up = thin % 2 == 0  # alternate above / below so leaders never cross a label
            ax.annotate(f"{col}: {s:.1f}% ({int(n):,})",
                        xy=(x, i - 0.3 if up else i + 0.3),
                        xytext=(x, i - 0.42 if up else i + 0.42),
                        ha="right" if x > 80 else "left" if x < 20 else "center",
                        va="bottom" if up else "top", fontsize=8, color="0.25",
                        clip_on=False, arrowprops=dict(arrowstyle="-", lw=0.7, color="0.5"))
            thin += 1
        left += s

ax.set_yticks([n * STEP for n in range(len(share))])
ax.set_yticklabels(share.index)
ax.set(xlim=(0, 100), ylim=((len(share) - 1) * STEP + 0.95, -0.95),
       xlabel="Share of the group's docks (%)", ylabel="",
       title="Live system-wide availability")
ax.grid(axis="y", visible=False)
ax.legend(ncol=3, loc="upper center", bbox_to_anchor=(0.5, -0.26), frameon=False, fontsize=11)
plt.tight_layout()
plt.show()

summary = parts.assign(**{"Total docks": parts.sum(axis=1)})
summary.insert(0, "Stations", [len(d) for d in groups.values()])
print(summary.astype(int).to_string())

n_inactive = len(groups["Inactive stations"])
inactive_docks = int(parts.loc["Inactive stations"].sum())
print(f"\nInactive stations              : {n_inactive:,} of {len(stations):,} "
      f"({100 * n_inactive / len(stations):.1f}%), holding {inactive_docks:,} docks "
      f"({100 * inactive_docks / parts.loc['Whole network'].sum():.1f}% of capacity)")

bikes = stations["num_bikes_available"].sum()
print(f"E-bikes among available bikes  : "
      f"{100 * stations['num_ebikes'].sum() / bikes:.0f}% of {int(bikes):,}")
print(f"Unreported docks               : {int(parts.loc['Whole network', 'Unreported']):,} "
      f"network-wide, of which {int(parts.loc['Inactive stations', 'Unreported']):,} "
      f"at inactive stations")
print(f"Bikes left at inactive stations: "
      f"{int(parts.loc['Inactive stations', ['Classic bikes', 'E-bikes']].sum()):,} rentable, "
      f"{int(parts.loc['Inactive stations', 'Bikes out of service']):,} out of service")

Approximately half of system docks hold rentable bikes, with e-bikes constituting a growing majority. With the network operating above 50% capacity, available bikes outnumber open docks. Out-of-service bikes occupy a small subset of docks without generating supply, whereas out-of-service docks remain rare.

The remaining capacity represents an "unreported" reconciliation gap (Section 2.4). Categorizing by station status shows that active stations balance cleanly, while inactive stations declare capacity while returning zero counts. This discrepancy is an artifact of the feed's registry structure rather than an operational miscount. Furthermore, inactive stations occasionally retain out-of-service bikes docked before removal from service.

### 2.4 Data Quality

#### Unreported Capacity

The grey slice in section 2.3 is the share of the network's docks that **no live counter accounts for**, and it is the largest quality issue in this dataset.
The plot below resolves the gap per station, in order to better understand its distribution.

In [ ]:
# The dock identity: bikes available + free docks + bikes out of service + docks out of
# service should equal capacity. Whatever is left over is capacity that no counter reports.
counted = (stations["num_bikes_available"] + stations["num_docks_available"]
           + stations["num_bikes_disabled"] + stations["num_docks_disabled"])
unreported = stations["capacity"] - counted  # > 0 docks nobody reports, < 0 counters above capacity
act, inact = stations["active"], ~stations["active"]
short = unreported > 0

print(f"Unreported docks (capacity - counters) : {int(unreported[short].sum()):,}")
print(f"  at inactive stations : {int(unreported[inact & short].sum()):,} "
      f"over {int((inact & short).sum())} stations")
print(f"  at active stations   : {int(unreported[act & short].sum()):,} "
      f"over {int((act & short).sum())} stations "
      f"(median {unreported[act & short].median():.0f}, max {int(unreported[act & short].max())} docks)")
print(f"  counters > capacity  : {int(-unreported[unreported < 0].sum()):,} docks "
      f"over {int((unreported < 0).sum())} stations")

# Per-station distribution, over the stations that declare any dock at all.
sized = stations["capacity"] > 0
print(f"\nStations with capacity > 0 : {int(sized.sum()):,}")
print(f"  reconcile exactly       : {int((unreported[sized] == 0).sum()):,} "
      f"({100 * (unreported[sized] == 0).mean():.1f}%)")
print(f"  within +/- 2 docks      : {100 * (unreported[sized].abs() <= 2).mean():.1f}%")
print(f"  largest single gap      : {int(unreported[sized].max())} docks unreported")

plt.figure(figsize=(8, 4))
sns.histplot(unreported[sized].clip(lower=-5, upper=15), bins=21)
plt.title("Unreported capacity per station")
plt.xlabel("Capacity - counted docks  (0 = consistent; tails folded at -5 / +15)")
plt.ylabel("Number of stations")
plt.tight_layout()
plt.show()

#### Identifier Consistency

The feed names a station **twice**, and the two identifiers are not interchangeable: `station_id` is internal to GBFS, while `short_name` is the public code of the station. Which of the two the trip files carry decides whether the live metadata can be joined to the ride history at all, so the choice is **verified below rather than assumed**.

In [ ]:
# Which identifier joins the two datasets? The trips carry only one of the feed's two.
trip_ids = pd.concat([df["start_station_id"], df["end_station_id"]]).dropna().astype(str)
distinct = set(trip_ids.unique())
short_names = set(stations["short_name"].astype(str))
gbfs_ids = set(stations["station_id"].astype(str))

print(f"Distinct station ids in the trips : {len(distinct):,}")
print(f"  matching short_name : {len(distinct & short_names):,} "
      f"({100 * len(distinct & short_names) / len(distinct):.1f}%)")
print(f"  matching station_id : {len(distinct & gbfs_ids):,}")

# station_id is not even uniform in shape, which is part of why it is unusable as a public key.
is_uuid = stations["station_id"].str.fullmatch(r"[0-9a-f]{8}-[0-9a-f]{4}-.*")
print(f"\nstation_id shaped as a UUID : {int(is_uuid.sum()):,} / {len(stations):,} "
      f"(the rest are numeric strings)")

# What the unmatched trip ids are made of.
unmatched = distinct - short_names
truncated = {i for i in unmatched if re.fullmatch(r"\d+\.\d", i) and f"{i}0" in short_names}
non_station = {i for i in unmatched if not re.fullmatch(r"[\d.]+", i)}
print(f"\nTrip ids absent from the feed : {len(unmatched):,}")
print(f"  truncated decimals, recovered by padding : {len(truncated)}")
print(f"  depots, system and demo entries          : {len(non_station)} "
      f"{sorted(non_station)[:4]}")
print(f"  retired stations                         : "
      f"{len(unmatched) - len(truncated) - len(non_station)}")

legs = int(trip_ids.isin(truncated).sum())
print(f"\nStation legs lost to the truncated ids : {legs:,} "
      f"({100 * legs / len(trip_ids):.2f}% of {len(trip_ids):,})")

# The station name cannot serve as a key either.
dup_names = stations["name"].value_counts()
dup_names = dup_names[dup_names > 1]
print(f"Names shared by more than one station  : {len(dup_names)} "
      f"{list(dup_names.index)}")

#### Missing Values

The table below is the proof that no field we consume is missing. Nulls alone would be weak evidence, since a missing value can hide as a sentinel, so every column is also checked for values that are present but cannot be real: a coordinate at the zero island or outside New York, a blank identifier, a negative count.

In [ ]:
NYC_LAT_RANGE, NYC_LON_RANGE = (40.4, 41.1), (-74.3, -73.6)

def suspect(col):
    """Values that are present but cannot be real."""
    s = stations[col]
    if col in ("lat", "lon"):
        lo, hi = NYC_LAT_RANGE if col == "lat" else NYC_LON_RANGE
        return int(((s == 0) | (s < lo) | (s > hi)).sum())   # zero island / outside NYC
    if s.dtype == bool:
        return 0
    if s.dtype.kind in "if":
        return int((s < 0).sum())                            # counts cannot be negative
    return int((s.astype(str).str.strip() == "").sum())      # blank text

completeness = pd.DataFrame({
    "dtype": stations.dtypes.astype(str),
    "nulls": stations.isna().sum(),
    "null %": (stations.isna().mean() * 100).round(2),
    "suspect values": [suspect(c) for c in stations.columns],
})
print(f"{len(stations):,} stations; per-column completeness:")
completeness

## 3. Weather Data

Hourly NYC weather from the [Open-Meteo archive API](https://open-meteo.com/), used to relate
ridership to weather.

In [ ]:
ride_year = df["ended_at"].min().year
start_date = date(ride_year, 1, 1)
end_date = min(date(ride_year, 12, 31), date.today() - timedelta(days=1))

# Ask for UTC and localise here rather than passing timezone=America/New_York
resp = requests.get(
    WEATHER_API_URL,
    params={
        "latitude": NYC_LAT,
        "longitude": NYC_LON,
        "start_date": (start_date - timedelta(days=1)).isoformat(),
        "end_date": (end_date + timedelta(days=1)).isoformat(),
        "hourly": "temperature_2m,precipitation,weather_code,wind_speed_10m",
        "timezone": "UTC",
        "wind_speed_unit": "kmh",
    },
    timeout=(5, 120),
)
resp.raise_for_status()

weather = pd.DataFrame(resp.json()["hourly"])
weather["datetime"] = (pd.to_datetime(weather["time"], utc=True)
                       .dt.tz_convert(NYC_TZ).dt.tz_localize(None))
weather = (weather[weather["datetime"].dt.date.between(start_date, end_date)]
           .drop(columns="time").reset_index(drop=True))
print(f"Fetched {len(weather):,} hourly rows: {start_date} -> {end_date}")
weather.head()

### 3.1 Structure and Summary

In [ ]:
weather.info()
print("\nSummary statistics:")
weather[["temperature_2m", "wind_speed_10m", "precipitation", "weather_code"]].describe()

### 3.2 Ridership vs. Weather

We join the hourly weather data with the trip data to analyze how weather conditions affect ridership patterns. The analysis focuses on key weather variables such as temperature and precipitation, and their correlation with the number of trips taken.

This helps identify how environmental factors influence bike usage, providing insights for operational planning and demand forecasting.

In [ ]:
# Join hourly ride counts to the matching weather hour.
df["hour_ts"] = df["ended_at"].dt.floor("h")
rides_per_hour = df.groupby("hour_ts").size().reset_index(name="rides")
merged = rides_per_hour.merge(weather, left_on="hour_ts", right_on="datetime", how="inner")
print(f"{len(merged):,} hours joined "
      f"({merged['hour_ts'].min():%Y-%m-%d} -> {merged['hour_ts'].max():%Y-%m-%d})")

# Daily granularity
daily = merged.set_index("hour_ts").resample("D").agg(
    rides=("rides", "sum"),
    temperature=("temperature_2m", "mean"),
    precipitation=("precipitation", "sum"))

# Small multiples on a shared date axis
RIDES_C, TEMP_C, PRECIP_C = "#2a78d6", "#eb6834", "#1baf7a"
fig, axes = plt.subplots(3, 1, figsize=(11, 8), sharex=True)

axes[0].plot(daily.index, daily["rides"], color=RIDES_C, lw=2, marker="o", ms=4)
axes[0].set_ylabel("Rides per day", color=RIDES_C)
axes[0].set_title("Daily ridership")

axes[1].plot(daily.index, daily["temperature"], color=TEMP_C, lw=2, marker="o", ms=4)
axes[1].set_ylabel("Mean temperature (°C)", color=TEMP_C)
axes[1].set_title("Daily mean temperature")
axes[1].axhline(0, color="0.6", lw=1, ls=":", zorder=0)

# Precipitation is a daily accumulation that is zero on most days
axes[2].bar(daily.index, daily["precipitation"], color=PRECIP_C, width=0.7)
axes[2].set_ylabel("Precipitation (mm/day)", color=PRECIP_C)
axes[2].set_title("Daily precipitation")

# The report claims the least-ridden day is also the wettest, so check it rather than assume it.
worst = daily["rides"].idxmin()
wettest = daily["precipitation"].idxmax()
for ax in axes:
    ax.axvline(worst, color="0.35", lw=1.2, ls="--", zorder=0)
axes[0].annotate(f"{worst:%b %d}: {daily.loc[worst, 'precipitation']:.0f} mm rain"
                 f"{' (month peak)' if worst == wettest else ''},\n"
                 f"lowest ridership of the month",
                 xy=(worst, daily.loc[worst, "rides"]), xytext=(58, 26),
                 textcoords="offset points", ha="center", fontsize=8.5,
                 bbox=dict(boxstyle="round,pad=0.3", fc="white", ec="0.7", alpha=0.9),
                 arrowprops=dict(arrowstyle="->", color="black", lw=1))

axes[2].xaxis.set_major_formatter(mdates.DateFormatter("%b %d"))
axes[2].set_xlabel("Day")
fig.autofmt_xdate()
plt.tight_layout()
plt.show()

print(f"least-ridden day = {worst:%b %d}, wettest day = {wettest:%b %d} "
      f"-> {'the same day' if worst == wettest else 'different days'}")
print(f"corr(rides, temperature)   = {daily['rides'].corr(daily['temperature']):+.3f}")
print(f"corr(rides, precipitation) = {daily['rides'].corr(daily['precipitation']):+.3f}")
# Same weekday, one wet and one dry, to separate rain from the weekend effect.
print(daily.loc[[worst, worst + pd.Timedelta(days=7)]].round(1))

This justifies weather as an explanatory variable for demand, which is why this dataset is
part of the project.

### 3.3 Data Quality

The feed is a well known and properly maintained source, so the checks here are limited to
completeness and to the one handling detail that is easy to get wrong: the **timezone**. The
archive applies a single fixed UTC offset to a whole request, whichever one New York happens to
be on the day the request runs, so asking it for local time silently mislabels every date on the
far side of a DST boundary by an hour. We therefore request UTC and convert ourselves (section 3
above), which is why the expected row count below is DST-aware rather than a flat 24 hours per
day: the spring-forward day has 23 hours and the autumn fall-back day has 25.

The table reports the missing count and percentage for every raw column.

In [ ]:
raw_cols = ["datetime", "temperature_2m", "precipitation", "weather_code", "wind_speed_10m"]
missing = weather[raw_cols].isna().sum()
missing = (pd.DataFrame({"missing": missing, "pct (%)": (missing / len(weather) * 100).round(2)})
           .sort_values("missing", ascending=False))
# Nulls alone would not catch an absent row, so check the hour count as well. Counting local
# hours rather than days * 24 keeps this honest across the DST transitions.
expected_hours = len(pd.date_range(start_date, end_date + timedelta(days=1),
                                   freq="h", tz=NYC_TZ, inclusive="left"))
print(f"{len(weather):,} hourly rows loaded, {expected_hours:,} expected "
      f"({start_date} -> {end_date}); missing values per column:")
missing

## 4. Bike Lanes

Follows the technical analysis of NYC's designated bike-route network, published annually by NYC DOT on
[NYC OpenData](https://data.cityofnewyork.us/dataset/New-York-City-Bike-Routes/mzxg-pwib/about_data)

In [61]:
routes = pd.read_csv(BIKE_ROUTES_URL, low_memory=False)

# instdate/ret_date arrive as MM/DD/YYYY text; parse once here, every section below reads these.
routes["install_date"] = pd.to_datetime(routes["instdate"], format="%m/%d/%Y", errors="coerce")
routes["retire_date"] = pd.to_datetime(routes["ret_date"], format="%m/%d/%Y", errors="coerce")

print(f"{len(routes):,} segments x {routes.shape[1]} columns")
print("Columns:", routes.columns.tolist())
routes.head()

28,983 segments x 27 columns
Columns: ['the_geom', 'segmentid', 'bikeid', 'prevbikeid', 'status', 'boro', 'street', 'fromstreet', 'tostreet', 'onoffst', 'facilitycl', 'allclasses', 'bikedir', 'lanecount', 'ft_facilit', 'tf_facilit', 'ft2facilit', 'tf2facilit', 'instdate', 'ret_date', 'grnwy', 'gwsystem', 'gwsys2', 'spur', 'gwyjuris', 'install_date', 'retire_date']


,the_geom,segmentid,bikeid,prevbikeid,status,boro,street,fromstreet,tostreet,onoffst,facilitycl,allclasses,bikedir,lanecount,ft_facilit,tf_facilit,ft2facilit,tf2facilit,instdate,ret_date,grnwy,gwsystem,gwsys2,spur,gwyjuris,install_date,retire_date
0,MULTILINESTRING ((-74.192429667502 40.52174149...,2579,6562,NaN,Current,5,HYLAN BLVD,HOLTEN AV,LUTEN AV,ON,II,II,2,2,Curbside Buffered,Curbside Buffered,NaN,NaN,10/01/2007,NaN,Greenway,Staten Island Waterfront,NaN,Main Alignment,NYCDOT,2007-10-01,NaT
1,MULTILINESTRING ((-74.16031384424598 40.589027...,5033,4272,NaN,Current,5,MERRYMOUNT ST,RICHMOND HILL RD,ROCKLAND AV,ON,II,II,2,2,Conventional Buffered,Conventional Buffered,NaN,NaN,08/12/2021,NaN,NaN,NaN,NaN,NaN,NaN,2021-08-12,NaT
2,MULTILINESTRING ((-74.12631986979562 40.635252...,10186,2107,NaN,Current,5,CLOVE ROAD,RICHMOND TERR,FOREST AVE,ON,III,III,2,2,Shared,Shared,NaN,NaN,09/11/2015,NaN,NaN,NaN,NaN,NaN,NaN,2015-09-11,NaT
3,MULTILINESTRING ((-74.00973537112554 40.645666...,20716,942,NaN,Current,3,5 AV,23 ST,50 ST,ON,III,III,2,2,Shared,Shared,NaN,NaN,07/02/2013,NaN,NaN,NaN,NaN,NaN,NaN,2013-07-02,NaT
4,MULTILINESTRING ((-74.02089492413252 40.626542...,126857,951,NaN,Current,3,6 AVENUE,67 ST,FT HAMILTON PKWY,ON,III,III,2,2,Shared,Shared,NaN,NaN,06/29/2015,NaN,NaN,NaN,NaN,NaN,NaN,2015-06-29,NaT


### 4.1 Structure and Summary

In [62]:
routes.info()

print("\nFacility class (the highest class present on the segment):")
print(pd.crosstab(routes["facilitycl"], routes["status"], margins=True, margins_name="All"))
print(f"\nDistinct values ever taken by facilitycl : {sorted(routes['facilitycl'].unique())}")
print(f"Date range (install / retire)            : "
      f"{routes['install_date'].min():%Y-%m-%d} -> {routes['install_date'].max():%Y-%m-%d} / "
      f"{routes['retire_date'].min():%Y-%m-%d} -> {routes['retire_date'].max():%Y-%m-%d}")

<class 'pandas.DataFrame'>
RangeIndex: 28983 entries, 0 to 28982
Data columns (total 27 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   the_geom      28983 non-null  str           
 1   segmentid     28983 non-null  int64         
 2   bikeid        28983 non-null  int64         
 3   prevbikeid    5396 non-null   float64       
 4   status        28983 non-null  str           
 5   boro          28983 non-null  int64         
 6   street        28983 non-null  str           
 7   fromstreet    28983 non-null  str           
 8   tostreet      28983 non-null  str           
 9   onoffst       28983 non-null  str           
 10  facilitycl    28983 non-null  str           
 11  allclasses    28983 non-null  str           
 12  bikedir       28983 non-null  str           
 13  lanecount     28983 non-null  int64         
 14  ft_facilit    21887 non-null  str           
 15  tf_facilit    21689 non-null  str           
 1

### 4.2 The Network Map

The geometry is what this dataset contributes to the dashboard, so the first thing to look at is
the network itself. Segments are drawn in the dashboard's own facility-class palette, and only
the ones **in service** are kept, which is what the dashboard shows before the year slider is
touched.

In [ ]:
# Palette, wording and basemap all copied from the dashboard, so this figure and the running
# app read the same (frontend .../utils/bikeRoutesLayer.js and .../utils/mapConfig.js).
FACILITY_COLORS = {"I": "#2f7d4f", "II": "#1953d8", "III": "#c88a1a", "L": "#6e6a62"}
FACILITY_LABELS = {"I": "Protected", "II": "Conventional",
                   "III": "Shared Lane or Signed Route", "L": "Link"}
BASE_TILE_URL = "https://a.basemaps.cartocdn.com/light_all/{z}/{x}/{y}@2x.png"
NYC_BBOX = (-74.28, 40.47, -73.68, 40.93)   # lon_min, lat_min, lon_max, lat_max

WEB_MERCATOR_R = 6378137.0
WORLD = math.pi * WEB_MERCATOR_R

def to_mercator(lon, lat):
    """Lon/lat degrees -> Web Mercator metres, the projection the tiles are drawn in."""
    return (np.radians(lon) * WEB_MERCATOR_R,
            np.log(np.tan(np.pi / 4 + np.radians(lat) / 2)) * WEB_MERCATOR_R)

def fetch_basemap(bbox, zoom=11):
    """Stitch the Positron tiles covering bbox; returns the image and its Mercator extent."""
    lon_min, lat_min, lon_max, lat_max = bbox
    n = 2 ** zoom

    def tile_xy(lon, lat):
        lat_r = math.radians(lat)
        return (int((lon + 180) / 360 * n),
                int((1 - math.log(math.tan(lat_r) + 1 / math.cos(lat_r)) / math.pi) / 2 * n))

    x0, y0 = tile_xy(lon_min, lat_max)   # north-west tile
    x1, y1 = tile_xy(lon_max, lat_min)   # south-east tile

    tile_px, canvas = None, None
    for x in range(x0, x1 + 1):
        for y in range(y0, y1 + 1):
            img = Image.open(io.BytesIO(requests.get(
                BASE_TILE_URL.format(z=zoom, x=x, y=y), timeout=(5, 30)).content)).convert("RGB")
            if canvas is None:
                tile_px = img.size[0]
                canvas = Image.new("RGB", (tile_px * (x1 - x0 + 1), tile_px * (y1 - y0 + 1)))
            canvas.paste(img, ((x - x0) * tile_px, (y - y0) * tile_px))

    span = 2 * WORLD / n   # metres covered by one tile at this zoom
    return canvas, (-WORLD + x0 * span, -WORLD + (x1 + 1) * span,
                    WORLD - (y1 + 1) * span, WORLD - y0 * span)

_COORDS = re.compile(r"(-?\d+(?:\.\d+)?)\s+(-?\d+(?:\.\d+)?)")

def wkt_lines(wkt):
    """Each line of a (MULTI)LINESTRING as Mercator xy; WKT stores lon before lat."""
    lines = []
    for part in re.findall(r"\(([^()]+)\)", wkt):
        pts = np.array(_COORDS.findall(part), dtype=float)
        lines.append(np.column_stack(to_mercator(pts[:, 0], pts[:, 1])))
    return lines

current = routes[routes["retire_date"].isna()]
tiles, extent = fetch_basemap(NYC_BBOX)

fig, ax = plt.subplots(figsize=(9, 9.5))
ax.imshow(np.asarray(tiles), extent=extent, origin="upper", interpolation="bilinear")
# Least protected first, so the protected classes stay on top where facilities overlap.
for cls in ["L", "III", "II", "I"]:
    lines = [line for wkt in current.loc[current["facilitycl"] == cls, "the_geom"]
             for line in wkt_lines(wkt)]
    ax.add_collection(LineCollection(lines, colors=FACILITY_COLORS[cls], linewidths=0.9))

(x_min, x_max), (y_min, y_max) = to_mercator(np.array(NYC_BBOX[::2]), np.array(NYC_BBOX[1::2]))
ax.set(xlim=(x_min, x_max), ylim=(y_min, y_max), aspect="equal")
ax.set_axis_off()
ax.legend(handles=[Line2D([], [], color=FACILITY_COLORS[c], lw=3,
                          label=f"{FACILITY_LABELS[c]} ({(current['facilitycl'] == c).sum():,})")
                   for c in FACILITY_COLORS],
          loc="upper left", fontsize=13, facecolor="white", framealpha=.92)
ax.set_title(f"NYC bike network by facility class ({len(current):,} segments in service)",
             fontsize=17)
ax.annotate("© OpenStreetMap contributors © CARTO", xy=(0.99, 0.012),
            xycoords="axes fraction", ha="right", va="bottom", fontsize=8, color="0.3")
plt.tight_layout()
plt.show()

# Protection is not spread evenly: compare the class mix borough by borough.
BORO = {1: "Manhattan", 2: "Bronx", 3: "Brooklyn", 4: "Queens", 5: "Staten Island"}
mix = pd.crosstab(current["boro"].map(BORO), current["facilitycl"], normalize="index") * 100
print("Share of each borough's segments by facility class (%):")
print(mix.round(1).to_string())

The lanes alone redraw the city: the five boroughs, the waterfronts and the park loops are all
legible without a basemap, which is why this layer works as the dashboard's spatial backdrop.

Colour is what makes it more than a shape, because protection is **not** spread evenly. Manhattan
is the only borough where *Protected* is the majority class (54% of its segments); Brooklyn, at
26%, is closest to its opposite and leans on *Conventional* striping instead. Brooklyn, Queens and
Staten Island each leave roughly a quarter of their network as *Shared Lane or Signed Route*, which
is a marking and a sign rather than a facility. A rider crossing a borough line can therefore drop
a protection level without ever leaving the network, which is exactly the comparison the map exists
to support.

### 4.3 Network Changes per Year

Because retired facilities stay in the file, the network can be replayed year by year. Counting
`instdate` above the axis and `ret_date` below it gives the dashboard's history chart: how much
was built and how much left service, every year.

The word *removed* needs care here. The data dictionary warns that a facility is retired *"when a
bicycle facility is modified or removed, when changes are made to the street network base map, or
in other circumstances"*, and concludes that **"retired bicycle facilities do not inherently
indicate the removal of a bicycle facility"**. The check below quantifies how much of the red is
really a rebuild rather than a loss.

In [ ]:
# Diverging bars, matching the dashboard's history chart (installed above, removed below).
INSTALLED_C, REMOVED_C = "#1953d8", "#c2501a"
SENTINEL = pd.Timestamp("1900-01-01")

install_year, retire_year = routes["install_date"].dt.year, routes["retire_date"].dt.year
years = np.arange(int(install_year.min()), int(install_year.max()) + 1)
installed = install_year.value_counts().reindex(years, fill_value=0)
removed = retire_year.value_counts().reindex(years, fill_value=0)

fig, ax = plt.subplots(figsize=(11, 4.5))
ax.bar(years, installed, color=INSTALLED_C, width=.8, label="Installed")
ax.bar(years, -removed, color=REMOVED_C, width=.8, label="Removed")
ax.axhline(0, color=".3", lw=1)
ax.annotate(f"{(routes['install_date'] == SENTINEL).sum()} segments dated 01/01/1900\n"
            "(placeholder for an unknown install date)",
            xy=(1901, installed[1900]), xytext=(52, -78), textcoords="offset points",
            fontsize=13, bbox=dict(boxstyle="round,pad=0.3", fc="white", ec="0.7"),
            arrowprops=dict(arrowstyle="->", color="black", lw=1))
ax.set_xlabel("Year", fontsize=14)
ax.set_ylabel("Segments", fontsize=14)
ax.tick_params(labelsize=13)
ax.yaxis.set_major_formatter(lambda v, _: f"{abs(int(v)):,}")
ax.legend(loc="upper left", fontsize=13)
ax.set_title("Bike-route segments installed and removed per year", fontsize=15)
plt.tight_layout()
plt.show()

# Is a "removal" a loss or a rebuild? A rebuild leaves a new record on the same stretch of
# street, installed the very day the old one retired.
RANK = {"L": 0, "III": 1, "II": 2, "I": 3}   # ascending level of protection
retired = routes[routes["retire_date"].notna()]
pairs = (retired.reset_index(names="row")
         .merge(routes[["segmentid", "install_date", "facilitycl"]],
                on="segmentid", suffixes=("", "_new")))
pairs = pairs[pairs["retire_date"] == pairs["install_date_new"]]
# A rebuild can leave more than one new record (e.g. one per direction): score each retirement
# by the best class it came back as, so every retirement counts exactly once.
rebuilt = (pairs.sort_values("facilitycl_new", key=lambda s: s.map(RANK))
           .drop_duplicates("row", keep="last"))

print(f"Retirements               : {len(retired):,}")
print(f"  rebuilt the same day    : {len(rebuilt):,} ({len(rebuilt) / len(retired):.1%})")
print(f"  no same-day replacement : {len(retired) - len(rebuilt):,}")

delta = rebuilt["facilitycl_new"].map(RANK) - rebuilt["facilitycl"].map(RANK)
print(f"\nOf those rebuilds: {(delta > 0).mean():.0%} came back better protected, "
      f"{(delta == 0).mean():.0%} unchanged, {(delta < 0).mean():.0%} downgraded")
print("\nMost common class transitions on rebuild:")
print(rebuilt.groupby(["facilitycl", "facilitycl_new"]).size()
      .sort_values(ascending=False).head(5).to_string())

The network is overwhelmingly a **recent** build: 21,207 of the 28,983 records were installed from
2007 onward. The 2,893 records predating 1996 are a different kind of infrastructure altogether,
two thirds of them *Protected* and off-street: greenways, park drives and bridge crossings that
were absorbed into the bike network rather than built as bike lanes.

Removals are negligible before 2008 (52 records in total) and only become a routine part of the
record afterwards, and they are mostly **not** losses. Two thirds of retirements (66.9%) are matched
by a new record on the same stretch of street installed the very same day, and 71% of those come
back better protected, most often a conventional lane rebuilt as a protected one. Read literally,
the red bars look like the city dismantling lanes; read correctly, they are largely the upgrade
programme showing up as churn. The chart is worth keeping in this form because it is the shape the
dashboard shows, but its caption has to say what red means.

### 4.4 Data Quality

Nothing here is missing: every column we consume is fully populated and every geometry parses.
The problems are of a different kind, and each one is checked below against what the data
dictionary says the file should contain: **identifiers that do not identify what their names
suggest**, **dates that carry a sentinel**, and a **status field that contradicts those dates**.

#### Identifier Semantics

Neither identifier means what its name suggests, and both are easy to misuse as a row key.
`segmentid` is *"From LION 23c. A number [...] that identifies each segment of a street"*: it names
a **stretch of asphalt**, not a record, so the same value returns every time that stretch is
rebuilt. `bikeid` is *"a unique number that corresponds to **one or more** LION segments"*: it names
a **route**, so it is shared by every segment along it. That is precisely why the dashboard
highlights a whole corridor when one of its segments is hovered.

The field that does track a facility across revisions is `prevbikeid`, and it is one of the columns
the ingestion drops, so the lineage it encodes is not available downstream.

In [ ]:
for col in ["segmentid", "bikeid"]:
    print(f"{col:10s}: {routes[col].nunique():,} distinct values over {len(routes):,} rows "
          f"-> {'unique' if routes[col].is_unique else 'NOT unique'}")

reused = routes.groupby("segmentid").size()
print(f"\nsegmentid values carrying more than one record : {(reused > 1).sum():,} "
      f"({reused[reused > 1].sum():,} rows)")
print("A rebuilt stretch of street, seen through its segmentid:")
example = reused[reused > 1].index[0]
print(routes.loc[routes["segmentid"] == example,
                 ["segmentid", "bikeid", "status", "facilitycl", "instdate", "ret_date", "street"]]
      .to_string(index=False))

# The DB writes on (segmentid, installation_date, status), so collisions there are dropped at load.
key = ["segmentid", "instdate", "status"]
print(f"\nRows sharing the ingestion's key {tuple(key)} : "
      f"{len(routes) - len(routes.drop_duplicates(key)):,} dropped on load")

# bikeid groups segments into a route: this is the hover-highlight unit in the dashboard.
per_route = routes.groupby("bikeid").size()
print(f"\nSegments per bikeid: median {per_route.median():.0f}, max {per_route.max()} "
      f"(over {len(per_route):,} routes)")
print(f"prevbikeid, the only true revision link, is dropped by the ingestion: "
      f"{routes['prevbikeid'].notna().sum():,} rows carry one")

#### Dates and Status

`instdate` is never null, which makes the file look complete on a missing-value check, but the
dictionary describes it as *"the **approximate** or exact date"* and unknown dates are encoded as
the sentinel **01/01/1900** rather than left blank. That single value is the earliest date in the
file, and the dashboard derives its year-slider lower bound from exactly that, so one placeholder
stretches the control over a century of empty years.

The dictionary also states the invariant that ties status to the dates: *"If Status = 'Current',
then RetDate = &lt;Null&gt;. If Status = 'Retired', then RetDate = MM/DD/YYYY"*. Our year filter,
in the API and in the frontend, is written on the dates alone, so any record that breaks the
invariant silently survives into the present-day network.

In [ ]:
sentinel = routes["install_date"] == SENTINEL
print(f"Segments dated 01/01/1900   : {sentinel.sum():,} "
      f"({sentinel.sum() / len(routes):.1%}), of which "
      f"{(sentinel & routes['retire_date'].isna()).sum():,} still in service")
print(f"Earliest real install date  : "
      f"{routes.loc[~sentinel, 'install_date'].min():%Y-%m-%d}")
print(f"Year-slider span it implies : {SENTINEL.year} -> {routes['install_date'].max().year} "
      f"({routes['install_date'].max().year - SENTINEL.year + 1} years, "
      f"{routes.loc[~sentinel, 'install_date'].dt.year.nunique()} of them non-empty)")

# The dictionary's Status <-> RetDate invariant, checked in both directions.
broken_current = (routes["status"] == "Current") & routes["retire_date"].notna()
broken_retired = (routes["status"] == "Retired") & routes["retire_date"].isna()
print(f"\nCurrent records carrying a retirement date : {broken_current.sum()}")
print(f"Retired records with no retirement date    : {broken_retired.sum()}")
print(routes.loc[broken_retired,
                 ["segmentid", "bikeid", "status", "instdate", "ret_date", "street", "boro"]]
      .to_string(index=False))
print("\nThese are retired, but a date-based year filter keeps them in the present-day network.")

# And one record retired before it was installed.
backwards = routes["retire_date"] < routes["install_date"]
print(f"\nRecords retired before being installed : {backwards.sum()}")
print(routes.loc[backwards, ["segmentid", "bikeid", "status", "instdate", "ret_date", "street"]]
      .to_string(index=False))

#### Completeness

The table below is the evidence that nothing we consume is missing. Nulls on their own would be
weak proof for a geometry column, since an unusable geometry can be present and still be
unparseable or in the wrong place, so `the_geom` is additionally parsed and bounds-checked
against New York.

In [ ]:
consumed = ["the_geom", "segmentid", "bikeid", "status", "boro", "street", "fromstreet",
            "tostreet", "facilitycl", "instdate", "ret_date"]
missing = routes[consumed].isna().sum()
completeness = pd.DataFrame({
    "nulls": missing,
    "null %": (missing / len(routes) * 100).round(2),
})
print(f"{len(routes):,} segments; per-column completeness "
      "(ret_date is null by design on records still in service):")
print(completeness.to_string())

# Geometry is the one column where "present" is not the same as "usable": read the raw lon/lat
# pairs back out of the WKT (not the projected ones the map uses) and bounds-check them.
raw_points = routes["the_geom"].map(lambda w: np.array(_COORDS.findall(w), dtype=float))
too_short = raw_points.map(len) < 2
outside = raw_points.map(lambda p: bool(
    ((p[:, 0] < -74.3) | (p[:, 0] > -73.6) | (p[:, 1] < 40.4) | (p[:, 1] > 41.1)).any()))
print(f"\nGeometries that fail to parse into a line : {int(too_short.sum())}")
print(f"Geometries with a point outside NYC       : {int(outside.sum())}")
print(f"Geometry types present                    : "
      f"{routes['the_geom'].str.split('(').str[0].str.strip().unique().tolist()}")

# facilitycl is the HIGHEST class on a segment, so a single colour can hide a mixed stretch.
mixed = routes["allclasses"].str.contains(",")
print(f"\nSegments carrying more than one facility class : {int(mixed.sum()):,} "
      f"({mixed.mean():.1%}) - the map shows only the best one")